# Gradient boosting, end to end

The three libraries the topic is named after, on the same dataset and split as the other notebooks, configured identically on purpose so the comparison is about the libraries rather than the settings.

The structural difference from a random forest is that boosting builds trees *in sequence*, each one correcting what the last got wrong. That is why it can overfit if you let it run, and why the tree count is chosen by early stopping rather than set by you. You set the ceiling; the validation slice decides where to stop.

In [1]:
import time

import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# 1. Same split as the other notebooks, plus a validation slice for early stopping
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# 2. The three libraries, configured the same way on purpose
results = {}

start = time.perf_counter()
xgb = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=3,
                    early_stopping_rounds=30, eval_metric="logloss")
xgb.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
results["XGBoost"] = (xgb.best_iteration, accuracy_score(y_test, xgb.predict(X_test)),
                      time.perf_counter() - start)

start = time.perf_counter()
lgbm = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, max_depth=3, verbose=-1)
lgbm.fit(X_fit, y_fit, eval_X=X_val, eval_y=y_val,
         callbacks=[lgb.early_stopping(30, verbose=False)])
results["LightGBM"] = (lgbm.best_iteration_, accuracy_score(y_test, lgbm.predict(X_test)),
                       time.perf_counter() - start)

start = time.perf_counter()
cat = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=3,
                         early_stopping_rounds=30, verbose=0)
cat.fit(X_fit, y_fit, eval_set=(X_val, y_val))
results["CatBoost"] = (cat.get_best_iteration(), accuracy_score(y_test, cat.predict(X_test)),
                       time.perf_counter() - start)

# 3. Early stopping chose the tree count; you only set the ceiling
print(f"{'library':10} {'trees used':>11} {'test acc':>9} {'fit sec':>8}")
for name, (rounds, acc, secs) in results.items():
    print(f"{name:10} {rounds:>11} {acc:>9.4f} {secs:>8.2f}")

library     trees used  test acc  fit sec
XGBoost             75    0.9561     0.35
LightGBM            80    0.9474     0.03
CatBoost           110    0.9649     0.49

## What the output is telling you

- **Nobody used the 1000 trees they were allowed.** Early stopping halted at 75, 80 and 110. `n_estimators` is a budget, not a target — the validation slice decides when the next tree stops helping.
- **LightGBM is roughly an order of magnitude faster** than the other two here, which is its whole reason for existing. CatBoost is slowest and, on this data, most accurate. The wall-clock numbers will differ on your machine; the ratio is the durable part.
- **All three lost to logistic regression's 0.9737.** Boosting is built for large, messy, interaction-heavy tabular data. Handed 455 clean rows with a nearly linear boundary, it has nothing to exploit and pays for the flexibility.

## When to reach for this

Gradient boosting is the default winner on substantial tabular problems — tens of thousands of rows and up, mixed feature types, interactions you cannot enumerate. It is what most tabular Kaggle solutions are built from, and it usually beats a forest given tuning.

Which of the three: **LightGBM** when rows or columns run large and training time is the constraint; **CatBoost** when you have high-cardinality categorical columns, which it encodes properly without manual work; **XGBoost** when you want the most documented, most portable option. Reach for none of them on small data, or when you need a model someone can read.

## Extend this notebook

- Drop `early_stopping_rounds` and let all 1000 trees run. Watch test accuracy fall while training accuracy climbs.
- Raise `learning_rate` to 0.3 and see early stopping trigger far sooner, at a worse score.
- Give CatBoost a genuinely categorical column and compare against one-hot encoding the others.
- Tune `max_depth` and `learning_rate` together — they trade off, so searching them one at a time finds the wrong pair.